# 03. 얼굴 분석 — 모델 여러 개를 잇는 파이프라인

> 큰 모델 하나가 다 하는 대신, **작은 모델 여러 개를 이어서** 일을 나눈다.
> 얼굴 찾기 → 얼굴만 잘라서 → 나이·성별 / 감정 / 머리 방향 모델에 각각 넣는다.

### 이 시간에 배우는 것
1. 모델 설명서마다 다른 **입력 크기 · 출력 이름**에 맞추기
2. 첫 모델의 결과(상자)로 둘째 모델의 입력(잘라 낸 얼굴)을 만드는 법
3. 출력의 뜻 풀기: 회귀(나이), 분류 확률(성별 · 감정), 각도(머리 방향)
4. 왜 NPU 에 잘 맞는가 — 작고 입력 크기가 고정된 모델

### 전체 흐름
```
사진 ─▶ [얼굴 찾기 300x300] ─▶ 상자들 ─자르기(여유 20%)─▶ 얼굴 사진들
                                          ├─▶ [나이·성별 62x62] ─▶ 나이, 여/남 확률
                                          ├─▶ [감정 64x64]      ─▶ 5가지 확률
                                          └─▶ [머리 방향 60x60] ─▶ 좌우·위아래·갸웃 각도
```

| 모델 | 입력 | 출력 |
|---|---|---|
| face-detection-retail-0005 | 300 x 300 BGR | `[1,1,N,7]` (01 번) |
| age-gender-recognition-retail-0013 | 62 x 62 BGR | `age_conv3` (나이 ÷ 100), `prob` ([여, 남]) |
| emotions-recognition-retail-0003 | 64 x 64 BGR | 5가지 확률: 무표정 · 행복 · 슬픔 · 놀람 · 화남 |
| head-pose-estimation-adas-0001 | 60 x 60 BGR | `angle_y_fc` · `angle_p_fc` · `angle_r_fc` (도) |

## 1. 준비 — 모델 네 개 올리기
`compile_model` 에 파일 경로를 바로 줘도 된다 (`read_model` 을 안에서 해 준다).
출력이 여러 개인 모델은 **출력 이름**으로 골라 꺼낸다 — 그래서 이름을 먼저 찍어 본다.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "..")).resolve()          # 프로그램 폴더 (lab.bat 이 알려 준다)
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # 모델 폴더
core = ov.Core()
print("모델 폴더:", MODELS)
print("장치:", core.available_devices)

def pick(*prefer):
    """먼저 적은 장치부터, 이 PC 에 있는 것을 고른다."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

DEVICE = pick("NPU", "GPU", "CPU")                  # 작은 고정 크기 모델들 — NPU 가 잘 맞는다
F = MODELS / "face"
det = core.compile_model(F / "face-detection-retail-0005.xml", DEVICE)
agegen = core.compile_model(F / "age-gender-recognition-retail-0013.xml", DEVICE)
emo = core.compile_model(F / "emotions-recognition-retail-0003.xml", DEVICE)
pose = core.compile_model(F / "head-pose-estimation-adas-0001.xml", DEVICE)
print("장치:", DEVICE)
for name, c in [("얼굴", det), ("나이·성별", agegen), ("감정", emo), ("머리 방향", pose)]:
    print(f"  {name:6s} 입력 {c.input(0).shape}  출력 {[o.any_name for o in c.outputs]}")

## 2. 사진

In [ ]:
PHOTO = "sample.jpg"          # 웹캠이 없을 때 쓸 사진 (이 노트북과 같은 폴더)

def show(bgr, width=720):
    """OpenCV 사진(BGR)을 노트북 안에 보여 준다."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """웹캠에서 한 장. 웹캠이 없으면 PHOTO 파일."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # 처음 몇 장은 어둡게 나오는 웹캠이 있다
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"웹캠도 {PHOTO} 도 없어요 — PHOTO 에 사진 파일 이름을 넣으세요"
        print("웹캠이 없어 파일을 씁니다:", PHOTO)
    return frame

frame = grab()
print("사진 크기(세로, 가로, 채널):", frame.shape)
show(frame)

## 3. 얼굴 찾기 → 잘라 내기
01 번과 같다. 하나 다른 점: 상자를 **조금 넉넉히(margin)** 잘라 낸다. 뒤의 모델들은 머리카락 · 턱까지 보이는 얼굴 사진으로 배웠기 때문이다.
너무 빡빡하게 자르면 나이 · 감정이 틀어진다.

In [ ]:
def blob(bgr, size):
    """세 모델 공통 전처리: 정사각형으로 크기만 바꾸고 BGR · 0~255 · NCHW."""
    return cv2.resize(bgr, (size, size)).transpose(2, 0, 1)[None].astype(np.float32)

def find_faces(bgr, conf=0.5, margin=0.2):
    h, w = bgr.shape[:2]
    dets = det(blob(bgr, 300))[det.output(0)].reshape(-1, 7)
    out = []
    for image_id, _, score, x1, y1, x2, y2 in dets:
        if image_id < 0:
            break
        if score < conf:
            continue
        x1, y1, x2, y2 = x1 * w, y1 * h, x2 * w, y2 * h
        mx, my = (x2 - x1) * margin, (y2 - y1) * margin        # 상자 크기의 margin 만큼 넉넉히
        box = (int(max(0, x1 - mx)), int(max(0, y1 - my)), int(min(w, x2 + mx)), int(min(h, y2 + my)))
        out.append((box, float(score)))
    return out

found = find_faces(frame)
print("얼굴", len(found), "개")

In [ ]:
# 각 모델이 실제로 '보는' 얼굴 — 크기를 맞춘 뒤 다시 키워서 보여 준다 (작을수록 흐릿하다)
for (x1, y1, x2, y2), s in found[:3]:
    face = frame[y1:y2, x1:x2]
    views = [cv2.resize(cv2.resize(face, (n, n)), (160, 160), interpolation=cv2.INTER_NEAREST) for n in (62, 64, 60)]
    print(f"얼굴 {face.shape[1]}x{face.shape[0]} → 나이·성별 62 · 감정 64 · 머리 방향 60")
    show(np.hstack(views), 480)

## 4. 잘라 낸 얼굴마다 세 모델

**출력 풀기**
- **나이** — 숫자 하나(회귀). 학습할 때 나이를 100 으로 나눠 배웠으므로 **× 100**.
- **성별** — `[여, 남]` 두 확률(합이 1). 큰 쪽을 고른다.
- **감정** — 다섯 확률. 가장 큰 것(`argmax`)이 답이고, 확률 자체도 의미가 있다 ("행복 0.55, 무표정 0.40" 은 애매하다는 뜻).
- **머리 방향** — 세 각도(도).

```
      위아래(pitch)            좌우(yaw)              갸웃(roll)
          ↑                    ←  ○  →                   ╱○
          ○  고개 끄덕         고개 도리도리              고개 기울이기
          ↓
```

In [ ]:
EMO = ["무표정", "행복", "슬픔", "놀람", "화남"]

def analyze(face):
    r = agegen(blob(face, 62))
    age = float(r[agegen.output("age_conv3")].squeeze()) * 100       # 출력 이름으로 꺼낸다
    female, male = r[agegen.output("prob")].squeeze()

    p = emo(blob(face, 64))[emo.output(0)].squeeze()                 # 다섯 확률

    r = pose(blob(face, 60))
    yaw = float(r[pose.output("angle_y_fc")].squeeze())      # 좌우 (도)
    pitch = float(r[pose.output("angle_p_fc")].squeeze())    # 위아래
    roll = float(r[pose.output("angle_r_fc")].squeeze())     # 갸웃
    return {"나이": round(age), "성별": "여" if female > male else "남", "성별 확신": round(float(max(female, male)), 2),
            "감정": EMO[int(p.argmax())], "감정 확률": np.round(p, 2),
            "좌우": round(yaw), "위아래": round(pitch), "갸웃": round(roll)}

view = frame.copy()
results = []
for i, ((x1, y1, x2, y2), s) in enumerate(found):
    info = analyze(frame[y1:y2, x1:x2])
    results.append(info)
    print(f"얼굴 {i}:", info)
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(view, f"#{i} age {info['나이']} yaw {info['좌우']}", (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
show(view)

`cv2.putText` 는 한글을 못 쓴다 — 그래서 사진에는 영어 · 숫자만 쓰고 한글은 글로 찍었다. (한글을 사진에 쓰려면 PIL 의 `ImageDraw.text` 와 한글 글꼴을 쓴다.)

### 감정 확률을 막대로

In [ ]:
import matplotlib.pyplot as plt     # 그래프 — 글자는 영어로 쓴다 (한글은 글꼴 설정이 따로 필요하다)

if results:
    p = results[0]["감정 확률"]
    plt.figure(figsize=(5, 2.4))
    plt.bar(["neutral", "happy", "sad", "surprise", "angry"], p, color="#1f5f7a")
    plt.ylim(0, 1); plt.title("face 0 emotion"); plt.show()

## 5. 머리 방향을 말로 — 규칙 만들기
각도는 숫자라 그대로는 쓰기 어렵다. **경계값**을 정해 "정면 / 왼쪽 / 오른쪽 / 위 / 아래" 로 바꾼다.
경계값(여기서는 15도)은 직접 해 보며 정한다. 카메라 위치 · 사람에 따라 달라진다.

In [ ]:
def facing(yaw, pitch, th=15):
    lr = "왼쪽" if yaw > th else ("오른쪽" if yaw < -th else "")
    ud = "위" if pitch < -th else ("아래" if pitch > th else "")
    return (lr + " " + ud).strip() or "정면"

for i, info in enumerate(results):
    print(f"얼굴 {i}: 좌우 {info['좌우']:+d}° · 위아래 {info['위아래']:+d}° → {facing(info['좌우'], info['위아래'])}")

각도의 **부호**(왼쪽이 + 인지 −인지)는 카메라가 거울처럼 뒤집혀 있는지에 따라 반대로 보일 수 있다. 직접 고개를 돌려 보며 맞춘다.

## 6. 얼굴 하나에 몇 ms? — 파이프라인 비용
모델을 이으면 시간도 더해진다. 얼굴이 5개면 분석 셋을 5번 돈다.

In [ ]:
if found:
    (x1, y1, x2, y2), _ = found[0]
    face = frame[y1:y2, x1:x2]
    for name, fn in [("찾기", lambda: find_faces(frame)), ("분석 셋", lambda: analyze(face))]:
        t0 = time.perf_counter()
        for _ in range(20):
            fn()
        print(f"{name:6s} {(time.perf_counter() - t0) / 20 * 1000:6.2f} ms  ({DEVICE})")

## 자주 나는 오류
| 증상 | 원인 · 고침 |
|---|---|
| `RuntimeError ... output ... not found` | 출력 이름이 다르다 → 1절에서 찍은 이름을 쓴다 |
| 나이가 이상함 (0~1 사이) | × 100 을 빠뜨렸다 |
| 얼굴이 사진 끝에 걸쳐 오류 | 자를 때 `max(0, …)` · `min(w, …)` 로 사진 안쪽으로 잘라야 한다 |
| 감정이 늘 '무표정' | 얼굴이 너무 작거나 빡빡하게 잘렸다 → `margin`, 가까이 |

## 정리
- 파이프라인 = 앞 모델의 **출력을 뒤 모델의 입력으로** 만드는 코드(자르기 · 크기 맞추기)가 핵심.
- 출력은 회귀(숫자) · 분류(확률) · 각도로 다르게 푼다. 확률은 "얼마나 확실한가" 도 알려 준다.
- 작은 고정 크기 모델 여러 개 → NPU 에 잘 맞는다.

## 해 볼 것
1. `margin` 을 0 과 0.5 로 바꾸면 나이 · 감정이 달라지나?
2. `DEVICE` 를 바꿔 6절 시간을 비교해 보자.
3. 웹캠을 여러 번 찍어(`frame = grab()`) 표정을 바꿔 가며 감정 확률이 어떻게 움직이는지 보자.
4. "정면을 3초 넘게 보면 알림" 같은 규칙을 만들어 보자.
   <details><summary>힌트</summary>

   웹캠을 열어 두고 반복하며 `facing(...) == "정면"` 이 이어진 시간을 잰다. 끊기면 시간을 0 으로.
   </details>

---
# 🛠 미니 프로젝트 — 집중도 측정기

**만들 것**: 20초 동안 웹캠을 보며, 장마다 "얼굴이 있고 정면을 보는가" 를 판정해 **집중 점수(정면 비율 %)** 를 내고
그동안의 **표정 분포**를 보여 준다. (온라인 수업 · 공부 타이머의 작은 판)

**쓰는 것**: `find_faces` · `analyze` · `facing` (이 노트북), `live`

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """웹캠 영상을 seconds 초 동안 process(사진) → 사진 으로 바꿔 노트북 안에 보여 준다. 멈추려면 I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # 웹캠이 없으면 사진으로
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n}장 · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (웹캠이 없어 사진으로 흉내)" if still is not None else ""))
    return out

In [ ]:
from collections import Counter
stats = {"장수": 0, "얼굴 있음": 0, "정면": 0}
moods = Counter()

def focus_meter(f):
    stats["장수"] += 1
    found = find_faces(f)
    msg, color = "no face", (0, 0, 255)
    if found:
        (x1, y1, x2, y2), _ = max(found, key=lambda it: (it[0][2] - it[0][0]) * (it[0][3] - it[0][1]))   # 가장 큰 얼굴
        info = analyze(f[y1:y2, x1:x2])
        stats["얼굴 있음"] += 1
        moods[info["감정"]] += 1
        d = facing(info["좌우"], info["위아래"])
        if d == "정면":
            stats["정면"] += 1; msg, color = "focus", (0, 200, 0)
        else:
            msg, color = "looking away", (0, 160, 255)
        cv2.rectangle(f, (x1, y1), (x2, y2), color, 2)
    score = stats["정면"] / stats["장수"] * 100
    cv2.putText(f, f"{msg}  score {score:.0f}%", (10, 34), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    return f

live(focus_meter, seconds=20)
print(f"집중 점수: {stats['정면'] / max(stats['장수'], 1) * 100:.0f}%  (얼굴 있음 {stats['얼굴 있음']}/{stats['장수']}장)")
print("표정 분포:", dict(moods.most_common()))

**더 해 보기**
- 5초 넘게 얼굴이 없으면 경고 문구 띄우기
- 점수를 1분마다 기록해 그래프로 (`matplotlib`)
- 경계값(`facing` 의 `th`)을 내 자리에 맞게 조정하기